## Classify the text, using LLM

In [7]:
import numpy as np
import pandas as pd
import sys
sys.path.append("..")

from typing import Literal
from pydantic import BaseModel, Field

In [36]:
class ChangeLabel(BaseModel):
    reason: str = Field(description="One sentence, at most 25 words, quoting the changed words")
    topic: Literal["revenue_demand", "costs_margins", "outlook_guidance", "capital_liquidity",
                   "credit_risk", "legal_regulatory", "macro", "strategy_mna", "accounting", "other"]
    materiality: Literal[0, 1, 2]
    direction: Literal["more_positive", "more_negative", "neutral_or_mixed", "not_applicable"]
    forward_looking: bool

PROMPT_VERSION = "v4"

SYSTEM_PROMPT = """\
You compare two versions of a paragraph from the MD&A section of a company's SEC filing
(10-Q or 10-K) and classify how the language changed, from the point of view of an investor.

## Input
Each request gives: ticker, pair type (yoy = same quarter last year, qoq = previous quarter,
annual = previous 10-K), the two periods, the section heading, and the change type:
- reworded: OLD and NEW are versions of the same paragraph
- added: only NEW exists (the paragraph is new in this filing)
- removed: only OLD exists (the paragraph was dropped)

Numbers have already been checked: a change that only updates numbers or dates is never sent.
Every item has a change in wording. Judge what the wording says. Use the numbers as context
for how large a change is, but a number moving is not by itself a reason to call a change material.

Most rewordings are small. Start from materiality 0 and direction not_applicable, and move up
only if you can name what an investor learns from NEW that OLD did not say.

These are always materiality 0:
- the explanation is unchanged and only numbers, dates or periods changed, including an added
  year-to-date (six- or nine-month) comparison next to the quarter
- cross-references ("refer to Note X", "see pages Y"), whatever they point to
- a dropped footnote or caveat that no longer applies
- grammar, punctuation, word order and formatting edits

## Fields
reason: one short sentence naming the key change, quoting the changed words. At most 25 words.

topic: the main subject of the changed content.
- revenue_demand: sales, volumes, pricing, customer demand, segment or product revenue
- costs_margins: operating expenses, cost of goods sold, gross margins, SG&A, interest expense, cost-saving initiatives
- outlook_guidance: only explicit company guidance or targets; otherwise pick the subject the statement is about, and mark forward_looking true
- capital_liquidity: debt, cash, buybacks, dividends, capital ratios, credit ratings, funding
- credit_risk: loan losses, allowances, charge-offs, credit quality of loans
- legal_regulatory: litigation, new compliance rules, government investigations, regulatory matters
- macro: inflation, interest rates, foreign exchange, supply chains, broader economic or industry trends
- strategy_mna: mergers, acquisitions, divestitures, strategic shifts, joint ventures, restructuring, capital projects
- accounting: accounting policies, restatements, critical accounting estimates, revenue recognition
- other: anything else, including business descriptions, cross-references and boilerplate

materiality:
- 0: an investor learns nothing new (see the list above)
- 1: new detail that does not change the picture (an extra driver named, a clarification)
- 2: changes the picture (a new or removed risk, changed guidance, a reversed trend, a new
     business driver, a new legal or regulatory matter, changed hedging about the future)

direction: is the company's situation or outlook described as better or worse than before?
- more_positive / more_negative / neutral_or_mixed / not_applicable
- For added paragraphs, judge the new information; for removed ones, the information dropped.
  A removed risk or problem is usually more_positive; an added one more_negative.
- neutral_or_mixed: the changed content has both positive and negative elements, or its impact
  is balanced or unclear. not_applicable: the change has no direction, such as factual
  background, neutral descriptions of past events, or edits with materiality 0.
- cannot be not_applicable at materiality = 1 or materiality = 2.

forward_looking: true if the paragraph is about the future (expectations, guidance, plans,
risks that may happen); false if it only describes the past or present. Judge the whole
paragraph (NEW, or OLD if removed), not only the words that changed.

## Consistency rules
- materiality 0 -> direction not_applicable.
- cross-references -> topic other.
"""
changes = pd.read_csv("../data/processed/changes_embedding.csv")

def user_message(row):
    lines = [
        f"Ticker: {row.ticker}",
        f"Pair type: {row.pair_type} ({row.old_period} -> {row.new_period})",
        f"Section: {row.new_section if pd.notna(row.new_section) else row.old_section}",
        f"Change type: {row.change}",
    ]

    if row.old_text:
        lines.append(f"OLD: {row.old_text}")
    if row.new_text:
        lines.append(f"NEW: {row.new_text}")

    return "\n".join(lines)

# The MS test item: hand-check row 13, "Fixed Income underwriting revenues decreased -> increased"
hc = pd.read_csv("../labels/diff_handcheck.csv")
target = hc.loc[13]
item = changes[(changes.ticker == target.ticker) & (changes.pair_type == target.pair_type)
               & (changes.old_period == target.old_period) & (changes.old_text == target.old_text)]
print(len(item))   # should be 1
print(user_message(item.iloc[0]))

1
Ticker: MS
Pair type: qoq (2025-06-30 -> 2025-09-30)
Section: Investment Banking Revenues
Change type: reworded
OLD: •Fixed Income underwriting revenues decreased primarily reflecting lower bond and investment-grade loan issuances, partially offset by higher non-investment grade loan issuances.
NEW: •Fixed Income underwriting revenues increased primarily reflecting higher non-investment grade and investment grade loan and bond issuances, which benefitted from higher event-related activity.


### Hand-annotated testcase

In [37]:
my_label = {
    "reason": "Fixed Income underwriting revenues shifted from a decrease to an increase, driven by higher bond and loan issuances.",            # your one sentence
    "change_kind": "explanation_changed",       # new_information | removed_information | explanation_changed | tone_or_hedging | trivial
    "topic": "revenue_demand",             # revenue_demand | costs_margins | outlook_guidance | capital_liquidity | credit_risk |
                             # legal_regulatory | macro | strategy_mna | accounting | other
    "materiality": 2,     # 0 | 1 | 2
    "direction": "more_positive",         # more_positive | more_negative | neutral_or_mixed | not_applicable
    "forward_looking": False, # True | False
}
ChangeLabel(**my_label)      # validates your label: a typo in a value raises an error

ChangeLabel(reason='Fixed Income underwriting revenues shifted from a decrease to an increase, driven by higher bond and loan issuances.', topic='revenue_demand', materiality=2, direction='more_positive', forward_looking=False)

In [29]:
import ollama

MODEL = "qwen3.5:9b"

response = ollama.chat(
    model=MODEL,
    messages=[{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": user_message(item.iloc[0])}],
    format=ChangeLabel.model_json_schema(),
    think=False,
    options={"temperature": 0, "seed": 42, "num_ctx": 8192, "num_predict": 300},
)

model_label = ChangeLabel.model_validate_json(response.message.content)
print(model_label.model_dump_json(indent=2))
print(f"prompt tokens {response.prompt_eval_count}, prefill {response.prompt_eval_duration / 1e9:.1f}s | "
      f"output tokens {response.eval_count}, decode {response.eval_duration / 1e9:.1f}s | "
      f"total {response.total_duration / 1e9:.1f}s")

{
  "reason": "Reversed trend description from 'decreased' to 'increased' and added driver 'higher event-related activity'.",
  "topic": "revenue_demand",
  "materiality": 2,
  "direction": "more_positive",
  "forward_looking": false
}
prompt tokens 983, prefill 0.2s | output tokens 70, decode 3.3s | total 3.5s


In [30]:
print("done_reason:", response.done_reason)          # "length" = cut off by num_predict
print("output tokens:", response.eval_count)
thinking = response.message.thinking or ""
print(len(thinking), "chars of thinking")
print("--- last 1,500 chars ---")
print(thinking[-1500:])

done_reason: stop
output tokens: 70
0 chars of thinking
--- last 1,500 chars ---



### Compare the o/p to the handwritten case

In [32]:
mine = ChangeLabel(**my_label)
for field in ["topic", "materiality", "direction", "forward_looking"]:
    a, b = getattr(mine, field), getattr(model_label, field)
    print(f"{field:16s} you: {str(a):22s} model: {str(b):22s} {'✓' if a == b else '✗'}")
print("\nmodel reason:", model_label.reason)

topic            you: revenue_demand         model: revenue_demand         ✓
materiality      you: 2                      model: 2                      ✓
direction        you: more_positive          model: more_positive          ✓
forward_looking  you: False                  model: False                  ✓

model reason: Reversed trend description from 'decreased' to 'increased' and added driver 'higher event-related activity'.


### Smoke test on 5 cases

In [33]:
def classify(row, model=MODEL):
    """One item through the model. Returns (ChangeLabel, raw response)."""
    response = ollama.chat(
        model=model,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_message(row)}],
        format=ChangeLabel.model_json_schema(),
        think=False,
        options={"temperature": 0, "seed": 42, "num_ctx": 8192, "num_predict": 300},
    )
    return ChangeLabel.model_validate_json(response.message.content), response

rng = np.random.default_rng(0)


def pick(mask):
    rows = changes[mask]
    return rows.iloc[rng.integers(len(rows))]

smoke = [
    ("added",           pick(changes.change == "added")),
    ("removed",         pick(changes.change == "removed")),
    ("likely trivial",  pick((changes.change == "reworded") & (changes.score >= 0.98))),
    ("JPM credit",      pick((changes.ticker == "JPM") & (changes.change == "reworded")
                             & changes.new_text.fillna("").str.contains("allowance for credit losses"))),
    ("forward-looking", pick((changes.change == "reworded")
                             & changes.new_text.fillna("").str.contains(r"\bexpect", case=False))),
]

# Show the inputs only: judge each one yourself before running cell 2
for i, (name, row) in enumerate(smoke):
    print(f"\n===== [{i}] {name} =====")
    print(user_message(row))


===== [0] added =====
Ticker: NVDA
Pair type: yoy (2025-04-27 -> 2026-04-26)
Section: Material Cash Requirements and Other Obligations
Change type: added
OLD: nan
NEW: For descriptions of our facility lease guarantees, long-term debt, purchase commitments, and operating lease obligations, refer to Note 8, Note 9, Note 10, and Note 14 of the Notes to the Condensed Consolidated Financial Statements in Part I, Item 1 of this Quarterly Report on Form 10-Q, respectively.

===== [1] removed =====
Ticker: JPM
Pair type: yoy (2025-03-31 -> 2026-03-31)
Section: Credit ratings
Change type: removed
OLD: (a)  The Moody's outlook of developing for JPMorgan Chase Bank, N.A. reflects its view with respect to possible support from the U.S. government, and its outlook of negative with an "(m)" modifier for J.P. Morgan SE reflects a negative outlook for long-term bank deposits and a positive outlook for the long-term issuer rating.
NEW: nan

===== [2] likely trivial =====
Ticker: KO
Pair type: qoq (202

In [34]:
results = []
for i, (name, row) in enumerate(smoke):
    label, resp = classify(row)
    secs = resp.total_duration / 1e9
    results.append({"name": name, **label.model_dump(), "seconds": round(secs, 1),
                    "prompt_tokens": resp.prompt_eval_count, "output_tokens": resp.eval_count})
    print(f"\n===== [{i}] {name}  ({secs:.1f}s, {resp.prompt_eval_count} in / {resp.eval_count} out) =====")
    for field, value in label.model_dump().items():
        print(f"  {field:16s} {value}")

import pandas as pd
smoke_df = pd.DataFrame(results)
print("\nmean seconds per item:", smoke_df.seconds.mean().round(1))


===== [0] added  (14.2s, 995 in / 61 out) =====
  reason           Added a cross-reference directing investors to specific notes for details on debt and lease obligations.
  topic            other
  materiality      0
  direction        not_applicable
  forward_looking  False

===== [1] removed  (8.0s, 990 in / 61 out) =====
  reason           The paragraph explaining Moody's specific outlook modifiers and government support views was dropped.
  topic            credit_risk
  materiality      0
  direction        not_applicable
  forward_looking  False

===== [2] likely trivial  (3.8s, 1157 in / 59 out) =====
  reason           Added six-month year-over-year comparison alongside the quarterly data.
  topic            costs_margins
  materiality      0
  direction        not_applicable
  forward_looking  False

===== [3] JPM credit  (4.0s, 972 in / 76 out) =====
  reason           The paragraph expanded from a single summary statement listing components to explicitly stating the provis